# Guided Project : Data Management with Databricks: Big Data with <img src="https://docs.delta.io/latest/_static/delta-lake-logo.png" width=300/>

<img src="https://upload.wikimedia.org/wikipedia/commons/9/97/Coursera-Logo_600x600.svg" width=50 height=50/>

**Project Scneario**: You are a Data Engineer working for an online clothing brand that sells a wide range of fashion Brands. The company's Supply Chain team has been tasked with building a dashboard to Analyze Orders history.

The supply chain team has been tasked with building a dashboard to **Analyze Orders history**. Your dashboard will be used to inform purchasing behaviour and ensure that the company has enough inventory to meet demand for the upcoming holiday season.

Throughout this real-world business scenario, you will learn how to create and ingest data into a delta table. Then use Databricks notebooks (using Python and SQL) to process/transform the data and produce the Supply chain dashboard. At the end you'll leverage Delta Lake's built-in functionalities such as merge operations and time travel to create a scalable data pipeline.

# TASK 2 - Upload project JSON files to Databricks file system

### b. Check loaded files

In [0]:
# Use Databricks Utilities (dbutils). Documentation : https://docs.databricks.com/dev-tools/databricks-utils.html#ls-command-dbutilsfsls 
dbutils.fs.ls("/Volumes/workspace/default/test_working/")

# TASK 3 - Create Delta Table : ORDERS_RAW

### a. Read multiline json files using spark dataframe:

In [0]:
# Import the necessary libraries
from pyspark.sql import SparkSession

# Create a SparkSession
spark = SparkSession.builder.appName("MultilineJSONRead").getOrCreate()

# Read the multiline JSON file with the 'multiline' option set to 'true' using spark dataframeAPI
orders_raw_df = spark.read.option("multiline", "true").json('/Volumes/workspace/default/orders/')

# Show the DataFrame and its schema (optional)
orders_raw_df.show(n=5,truncate=False)

## click on orders_raw_df to Check the schema
orders_raw_df.printSchema()

In [0]:
#Validate loaded files Count Number of Rows in the DataFrame, the total Should be "1510"
orders_raw_df.count()

### ![b.](https://pages.databricks.com/rs/094-YMS-629/images/delta-lake-tiny-logo.png) b. Create Delta Table ORDERS_RAW

Delta Lake is 100% compatible with Apache Spark&trade;, which makes it easy to get started with if you already use Spark for your big data workflows.
Delta Lake features APIs for **SQL**, **Python**, and **Scala**, so that you can use it in whatever language you feel most comfortable in.


   <img src="https://databricks.com/wp-content/uploads/2020/12/simplysaydelta.png" width=400/>

In [0]:
# First, Create Database SupplyChainDB if it doesn't exist
db = "SupplyChainDB"

spark.sql(f"CREATE DATABASE IF NOT EXISTS {db}")
spark.sql(f"USE {db}")

In [0]:
## Create DelaTable ORDERS_RAW in the metastore using DataFrame's schema and write data to it
## Documentation : https://docs.delta.io/latest/quick-start.html#create-a-table

# Create the table if it does not exist. Otherwise, replace the existing table.
# orders_raw_df.writeTo(f"workspace.{db}.orders_raw").createOrReplace()

# If you know the table does not already exist, you can use this command instead.
orders_raw_df.write.saveAsTable(f"workspace.{db}.orders_raw")

# View the new table.
orders_raw_df = spark.read.table(f"workspace.{db}.orders_raw")
display(orders_raw_df)

### c. Show Created Delta Table:

In [0]:
%sql
-- Switch to SQL Cell using %SQL
SHOW tables
 
 -- Alternativerly you can use Python: display(spark.sql(f"SHOW TABLES"))

**d. Validate data loaded successfully to Delta Table ORDERS_RAW**:

In [0]:
%sql
select * from workspace.supplychaindb.orders_raw limit 10


**e. Decsribe Detail of the Delta Table**:

In [0]:
%sql
-- describe DETAIL ORDERS_RAW
describe detail workspace.supplychaindb.orders_raw;
-- Returns the basic metadata information of a delta table.
-- describe table workspace.supplychaindb.orders_raw;

#Practice Activity 1 : Create INVENTORY Delta table

### a. Upload INVENTORY.Json file in DBFS

###b. Read the File using spark dataframe

In [0]:
inventory_df = spark.read.option("multiline", "true").json("/Volumes/workspace/default/inventory/")
inventory_df.printSchema()

## Show the datafarme
inventory_df.show(n=5, truncate=False)

### ![c.](https://pages.databricks.com/rs/094-YMS-629/images/delta-lake-tiny-logo.png) c. Create Delta Table INVENTORY

In [0]:
# First, Create Database SupplyChainDB
db = "SupplyChainDB"
spark.sql(f"USE {db}")

In [0]:
## Create INVENTORY Delta Table 
inventory_df.write.saveAsTable(f"workspace.{db}.inventory")

### d. Show Created Delta Tables:

In [0]:
%sql
-- Switch to SQL Cell using %sql
SHOW TABLES

# TASK 4 - Transform data in delta table

<a href="https://www.databricks.com/glossary/medallion-architecture" target="_blank">Medallion Architecture</a>   
</br>
<img src="https://databricks.com/wp-content/uploads/2020/09/delta-lake-medallion-model-scaled.jpg" width=900/>

During this Task you will : 
* 1- Read delta Table using Spark Dataframe
* 2- Convert Data Type String --> Date
* 3- Drop Rows with Null Values
* 4- Add a Computed Column "TOTAL_ORDER"
* 5- Create new deltatable Orders_Gold

### a. Read ORDERS_RAW delta table using spark Dataframe

In [0]:
#read Delta Table using spark dataframe

ORDERS_Gold_df= spark.read.table("workspace.supplychaindb.orders_raw")

ORDERS_Gold_df.show(n=5,truncate=False)
# Click on ORDERS_DF to See the Schema of the Table. 

### b. Update ORDER_DATE Column's Data Type

In [0]:
#Use withColumn method & to_date()
# withColumn Documentation : https://spark.apache.org/docs/3.1.3/api/python/reference/api/pyspark.sql.DataFrame.withColumn.html
# TO_DATE() Documentation : https://docs.databricks.com/sql/language-manual/functions/to_date.html

from pyspark.sql.functions import *
import pyspark.sql.functions as sf

ORDERS_Gold_df = ORDERS_Gold_df.withColumn("ORDER_DATE", to_date("ORDER_DATE","yyyy-MM-dd"))
# Use describe() method to get the summary statistics of the DataFrame

# ORDERS_Gold_df.show()
display(ORDERS_Gold_df.describe())

### c. Drop Rows with Null Values

In [0]:
# Count Nulls for each column
from pyspark.sql.functions import *

display(ORDERS_Gold_df.select([count(when(col(c).isNull(),c)).alias(c) for c in ORDERS_Gold_df.columns]))

In [0]:
#  Remove Nulls using dropna() method which removes all rows with Null Values 

ORDERS_Gold_df = ORDERS_Gold_df.dropna()

ORDERS_Gold_df.count()

### d. Add new Column TOTAL_ORDER

In [0]:
#Use withColumn function
#Documentation : https://spark.apache.org/docs/3.1.3/api/python/reference/api/pyspark.sql.DataFrame.withColumn.html


ORDERS_Gold_df= ORDERS_Gold_df.withColumn("TOTAL_ORDER", sf.expr("QUANTITY * UNIT_PRICE"))

# Display ORDERS_Gold_df to validate the creation of the New Column TOTAL_ORDER
display(ORDERS_Gold_df)

### e. Create Delta Table ORDERS_GOLD

In [0]:
# Make sure you are using SupplyChainDB
spark.sql(f"USE SupplyChainDB")

## Create DeltaTable Orders_GOLD: 

ORDERS_Gold_df.write.saveAsTable(f"workspace.supplychaindb.orders_gold")


## Validate that the table was created successfully
display(spark.sql(f"SHOW TABLES"))

In [0]:
display(spark.sql(f"SHOW TABLES"))

-- Read more about different write options and parameters here https://docs.delta.io/latest/delta-batch.html#write-to-a-table 

* **Append** to automatically add new data to an existing Delta table, 
* **Overwrite** To automatically replace all the data in a table:

# TASK 5 - Query Orders Delta table using SQL

### Get Familiar with Orders_Gold dataset

In [0]:
%sql
select * from workspace.supplychaindb.orders_gold
limit 30
-- Get top 30 rows Get Familiar with the Data


### KPI-1: Quantity Sold by Country

In [0]:
%sql
-- Division = CATEGORY 
-- Dont forget to Filter out Cancelled Orders
select sum(QUANTITY) as qnt_sum, 
-- round(sum(TOTAL_ORDER)) as total_sales,
 ORDER_COUNTRY
from workspace.supplychaindb.orders_gold 
where order_status != 'Cancelled' 
group by ORDER_COUNTRY
order by qnt_sum desc ;


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Division = CATEGORY 
-- Dont forget to Filter out Cancelled Orders
select sum(QUANTITY) as qnt_sum, sum(TOTAL_ORDER) as total_sales, CATEGORY
from workspace.supplychaindb.orders_gold 
where order_status != 'Cancelled' 
group by CATEGORY
order by qnt_sum desc ;


### KPI-2: Sales by Division ($)

In [0]:
%sql
-- Dont forget to Filter out Cancelled Orders
select sum(QUANTITY) as qnt_sum, round(sum(TOTAL_ORDER)) as total_sales, ORDER_COUNTRY
from workspace.supplychaindb.orders_gold 
where order_status != 'Cancelled' 
group by ORDER_COUNTRY
order by total_sales desc

In [0]:
%sql
-- Dont forget to Filter out Cancelled Orders
select sum(QUANTITY) as qnt_sum, round(sum(TOTAL_ORDER)) as total_sales, CATEGORY
from workspace.supplychaindb.orders_gold 
where order_status != 'Cancelled' 
group by CATEGORY
order by total_sales desc

Databricks visualization. Run in Databricks to view.

### KPI-3: Top-5 Popular Brands

In [0]:
%sql
-- Limit Result to 5 and Order Results and order by Sold Quanity
select sum(QUANTITY) as qnt_sum, round(sum(TOTAL_ORDER)) as total_sales, BRAND
from workspace.supplychaindb.orders_gold 
where order_status != 'Cancelled' 
group by brand
order by qnt_sum desc
limit 5


Databricks visualization. Run in Databricks to view.

# TASK 6 - Create Dashboard

In [0]:
# Use Databricks UI
# 1- Turn results of Previous Queries into visualisations
# 2- Create Dashboard and add Visualisations

# Practice Activity 2 : Add Monthly Sales Trend to your Dashboard

### KPI-4: Monthly Sales Trend (In QTY)

** Instructions :**  
  # 1- Query Delta Table: Orders_Gold to extract Monthly Sales (in Quantity, across all brands and all regions) 
  # 2- Turn the result (Table) into a visualisation (line chart) to Show the Trend for the last 18 months.
  # 3- Add your visualization to the Supply Chain Dashboard.

In [0]:
%sql
-- Use DATE_TRUNC()  
select DATE_TRUNC("month", ORDER_DATE) as month_od, sum(QUANTITY) as sum_qnt --, BRAND, ORDER_COUNTRY
from workspace.supplychaindb.orders_gold 
where order_status != 'Cancelled'
group by month_od 
-- group by BRAND, ORDER_COUNTRY, month_od
order by month_od


Databricks visualization. Run in Databricks to view.

# TASK 7 - Update Data in Orders table using Merge

<img src="https://databricks.com/wp-content/uploads/2020/09/delta-lake-medallion-model-scaled.jpg" width=1012/>

### a. Upload Json files into DBFS

Use UI to upload the file "UPDATE_ORDERS_RAW.json" into DBFS, use the same folder dbfs:/FileStore/SupplyChain/ORDERS_RAW/

### b. Read file using Spark dataframe

In [0]:
# Read multiple line json file UPDATE_ORDERS_RAW.json
Update_orders_df = spark.read.option("multiline", "true").json("/Volumes/workspace/default/orders/UPDATE_ORDERS_RAW.json")

## Show the datafarme
display(Update_orders_df)

-->Check the original data **BEFORE MERGE**

In [0]:
%sql 
select ORDER_ID,ORDER_STATUS,Quantity from Supplychaindb.ORDERS_RAW WHERE ORDER_ID in ("ORD-1281","ORD-829","ORD-193","ORD-826","ORD-842")

### c. Update Orders_RAW deltatable using Merge

In [0]:
%sql
DESCRIBE DETAIL supplychaindb.ORDERS_RAW

In [0]:
import delta.tables as dt
import pyspark

# programmatically interacting with Delta tables using the class delta.tables.DeltaTable(spark: pyspark.sql.session.SparkSession, jdt: JavaObject)

delta_orders_raw = dt.DeltaTable.forName(spark, "orders_raw")

# dbutils.fs.ls("/Volumes/workspace/default/orders")

# display(delta_orders_raw.detail())

In [0]:
## merge data into delta Table ORDER_RAW
# DOCUMENTATION https://docs.delta.io/latest/delta-update.html#language-python 

delta_orders_raw.alias("original").merge(
  Update_orders_df.alias("updates"),
  "updates.ORDER_ID = original.ORDER_ID"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

# must be at least one WHEN clause in a MERGE statement.

--> check the udaptes rows **AFTER MERGE**

In [0]:
%sql 
select ORDER_ID,ORDER_STATUS,Quantity from SUPPLYCHAINDB.ORDERS_RAW WHERE ORDER_ID in ("ORD-1281","ORD-829","ORD-193","ORD-826","ORD-842")

Learn More about Merge Operations check out https://docs.delta.io/latest/delta-update.html#language-python

# TASK 8 - Query previous versions of delta table using **Time Travel**

**This Task shows how to time travel between different versions of a Delta table with Delta Lake. You can time travel by table version or by timestamp. You’ll learn about the benefits of time travel and why it’s an essential feature for production data workloads.**

**Documentation : https://delta.io/blog/2023-02-01-delta-lake-time-travel/** 

<img src="https://delta.io/static/9c42ea9f028932de03257ed75d35a8ba/cf8e5/image1.png" width=1012/>

### a. Describe Detla Table History:

In [0]:
%sql
-- Check Table History 
describe history supplychaindb.ORDERS_RAW
-- Use the UI to see Delta Table History

### b. Using SQL:

In [0]:
%sql 
 select ORDER_ID,ORDER_STATUS,Quantity from SUPPLYCHAINDB.ORDERS_RAW VERSION AS OF 1 WHERE ORDER_ID in ("ORD-1281","ORD-829","ORD-193","ORD-826","ORD-842")

-- CHange Version Number to See different Versions of the delta table

### c. Using Spark dataframe:

In [0]:
#Time Travel
version_1 = spark.read.format('delta').option('TimeStamp', "2023-05-16").table("SUPPLYCHAINDB.ORDERS_RAW")
display(version_1)

# END OF THE PROJECT

# CUMULATIVE CHALLENGE

**Your Task :</br> Using the “Inventory” data, your task is to enrich the Supply Chain Dashboard with the list of low-stock and out-of-stock Items.** 

Using Databricks notebook you will : </br>
1-Upload INVENTORY.JSON file to DBFS(1) </br>
2-Read the file using spark dataframe (1)</br>
3-Create Delta Table INVENTORY (1)</br>
4-Write an SQL query to cross join ORDERS_GOLD and INVENTORY DeltaTables to find the list of Items Low-in Stock or Out-of Stock</br>
5-Turn the result into a Visualisation (Table type) and Add it to your SupplyChain Dashboard</br>

</br>(1) Skip if you have completed Practice Activity 1

### a. Upload INVENTORY.Json file in DBFS

In [0]:
## Load the file using the UI to this path dbfs:/FileStore/SupplyChain/INVENTORY/

### b. Read the File using spark dataframe

In [0]:
inventory_df = spark.read.option("multiline", "true").json("/Volumes/workspace/default/inventory/")
inventory_df.printSchema()

## Show the datafarme
inventory_df.show(n=5, truncate=False)

### c. Create Delta Table INVENTORY <img src="https://pages.databricks.com/rs/094-YMS-629/images/delta-lake-tiny-logo.png" width=35 height=35/>

In [0]:
# Use SupplyChainDB Database
db = "SupplyChainDB"
spark.sql(f"USE {db}")

In [0]:
## Create INVENTORY Delta Table 
# inventory_df.write.saveAsTable(f"workspace.{db}.inventory")

## Validate that the table was created successfully
display(spark.sql(f"SHOW TABLES"))

### d. Cross Join ORDERS_GOLD and INVENTORY DeltaTables to find the list of Low Stock or Out-of Stock Items

**Your Goal** is to find the list of Low-Stock or Out-of-Stock Items and Add the result to your SupplyChain Dashboard<br />
**Hints:**
* Group all Orders (from ORDERS_GOLD) based-on BRAND, COLOR, PRODUCT_NAME AND SIZE And Add QTY_SOLD (SUM QUANTITY) 
* Cross join the result with INVENTORY using an inner join on BRAND, PRODUCT_NAME, COLOR AND SIZE.
* Add calculated column "QTY_LEFT_STOCK" as (STOCK - QTY_SOLD)
* Filter-out Cancelled ORDERS (ORDER_STATUS)
* Keep only Items with QTY_LEFT_STOCK < 20
* Sort the result by "QTY_LEFT_STOCK" in ascending order

In [0]:
%sql
-- Write Your Query Here : 
with qty_sold as(
select og.BRAND, og.COLOR, og.PRODUCT_NAME, og.SIZE, SUM(og.QUANTITY) AS QTY_SOLD, 
(i.STOCK-qty_sold) AS QTY_LEFT_STOCK
from workspace.supplychaindb.orders_gold as og
inner join workspace.supplychaindb.inventory as i
ON og.BRAND = i.BRAND
AND og.PRODUCT_NAME = i.PRODUCT_NAME
AND og.COLOR = i.COLOR
AND og.SIZE = i.SIZE     
WHERE ORDER_STATUS != 'Cancelled' 
GROUP BY og.BRAND, og.COLOR, og.PRODUCT_NAME, og.SIZE, i.STOCK
order by QTY_LEFT_STOCK asc)

select * from qty_sold
where qty_left_stock < 20
;


Databricks visualization. Run in Databricks to view.

### e. Turn the result into a Visualisation (Table) and Add it to SupplyChain Dashboard

In [0]:
# Use Databricks UI to Turn results into a visualisation and then add it to your SupplyChain Dashboard

#  
   <img src="https://www.freeiconspng.com/uploads/congratulations-png-1.png" width=350/>
   .... THIS IS THE END OF THE GUIDED PROJECT